# Embedded-atom potentials

``matscipy`` implements support for embedded-atom method potentials. The class {py:class}`matscipy.calculators.eam.EAM` implements the functional form

\begin{equation}
  U
  =
  \sum_{i} F(\rho_i)
  +
  \frac{1}{2}
  \sum_{\substack{ij\\ i\neq j}}
  U_\text{rep}(r_{ij})
\end{equation}

with

\begin{equation}
  \rho_i
  = 
  \sum_{\substack{j\\ j\neq i}}
  f(r_{ij})
\end{equation}

as described, e.g. by [Müser et al.](https://doi.org/10.1080/23746149.2022.2093129). On top of energies and forces, the calculator can compute second derivatives (with respect to positions and strain degrees of freedom). The function $F(\rho)$ is the embedding function, $U_\text{rep}(r)$ is the repulsive potential, and $f(r)$ is a distance-dependent function used to compute the density.

## Cleri-Rosato potential

The following example computes the elastic constants of a small representation of cupper using the potential by [Cleri & Rosato](https://doi.org/10.1103/PhysRevB.48.22). We first create a small FCC cell.

In [64]:
from ase.build import bulk

a = bulk('Cu', 'fcc', a=3.544, cubic=True)
a *= (2, 2, 2)

def interactive_view(system):
    from ase.visualize import view
    # Interactive view of the lattice
    v = view(system, viewer='ngl')

    # Resize widget
    v.view._remote_call("setSize", target="Widget", args=["300px", "300px"])
    v.view.center()
    return v

interactive_view(a)

Now we setup the calculator with the EAM potential and its parametrization.

In [72]:
import numpy as np
from matscipy.calculators.eam import EAM

# Cleri-Rosato functional form
class F:
    def __call__(self, rho):
        return -np.sqrt(rho)
    
    def derivative(self, n):
        if n == 1:
            return lambda rho: -0.5 / np.sqrt(rho)
        elif n == 2:
            return lambda rho: 0.25 / (rho**(3/2))
        else:
            raise NotImplementedError("Higher order derivatives not implemented")

class f:
    def __init__(self, xi=1.224, q=2.278, r0=3.615/np.sqrt(2)):
        self.xi = xi
        self.q = q
        self.r0 = r0

    def __call__(self, r):
        return self.xi**2 * np.exp(-2*self.q*(r/self.r0-1))
    
    def derivative(self, n):
        if n == 1:
            return lambda r: -2*self.q*self.xi**2 * np.exp(-2*self.q*(r/self.r0-1)) / self.r0
        elif n == 2:
            return lambda r: 4*self.q**2*self.xi**2 * np.exp(-2*self.q*(r/self.r0-1)) / self.r0**2
        else:
            raise NotImplementedError("Higher order derivatives not implemented")

class rep:
    def __init__(self, A=0.0855, p=10.960, r0=3.615/np.sqrt(2)):
        self.A = A
        self.p = p
        self.r0 = r0

    def __call__(self, r):
        return self.A * np.exp(-self.p*(r/self.r0-1))
    
    def derivative(self, n):
        if n == 1:
            return lambda r: -self.p*self.A * np.exp(-self.p*(r/self.r0-1)) / self.r0
        elif n == 2:
            return lambda r: self.p**2*self.A * np.exp(-self.p*(r/self.r0-1)) / self.r0**2
        else:
            raise NotImplementedError("Higher order derivatives not implemented")

calc = EAM(atomic_numbers=[29], F=[F()], f=[[f()]], rep=[[rep()]], cutoff=8.0)
a.calc = calc

print(f'Cohesive energy = {a.get_potential_energy() / len(a):.2f} eV/atom')

Cohesive energy = -4.56 eV/atom


In [62]:
func = rep()
x = np.linspace(0.5, 1.0, 1000)
dx = x[1] - x[0]
numdiff = np.diff(func.derivative(1)(x)/dx)
anadiff = func.derivative(2)(x+dx/2)
np.max(abs(numdiff - anadiff[:-1]))

0.0009518316692265216

We can also compute the Born elastic constants, i.e. the affine elastic constants:

In [71]:
import numpy as np
from ase.units import GPa
from matscipy.elasticity import elastic_moduli, full_3x3x3x3_to_Voigt_6x6

# Born elastic constants (without nonaffine displacements)
C = a.calc.get_property('born_constants', a)

# Get isotropic elastic moduli
E, nu, G, B, K = elastic_moduli(full_3x3x3x3_to_Voigt_6x6(C))

display(Markdown(f"Young's modulus = {E.mean() / GPa:.1f} GPa"))
display(Markdown(f"Poisson number = {(nu + np.eye(3)).sum()/6:.3f}"))

PropertyNotImplementedError: born_constants property not implemented